<a href="https://colab.research.google.com/github/vnz626/DashboardAnalisisDataGTAVI/blob/main/proses_data_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proses Data: Postingan X tentang GTA 6

Notebook ini berisi:
1. Pengambilan data dari X dengan **tweet-harvest**
2. Pembersihan data → `data/gta6_clean.csv`
3. Sampel 20 postingan untuk **label manual** → `data/evaluasi_sentimen.csv`
4. Cek cepat hasil clustering dan regresi (untuk menentukan nama tema)

**Versi Google Colab.** Jalankan sel dari atas ke bawah. Di sel persiapan, unggah file `analysis.py` dari folder proyek supaya hasil di sini sama dengan aplikasi Streamlit.

## 0. Persiapan Colab

In [ ]:
# Pasang VADER dan Node.js 20 (dibutuhkan tweet-harvest)
!pip install -q vaderSentiment
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - > /dev/null 2>&1
!sudo apt-get install -y nodejs > /dev/null 2>&1
!node -v

v20.20.2


In [ ]:
# Unggah analysis.py dari folder proyek
import os
from google.colab import files
if not os.path.exists("analysis.py"):
    files.upload()
assert os.path.exists("analysis.py"), "analysis.py belum terunggah"


Saving analysis.py to analysis.py


In [ ]:
import sys, os, re, subprocess
from datetime import datetime, timezone
from getpass import getpass

import pandas as pd


sys.path.append(".")
import analysis as an

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Pengambilan Data (tweet-harvest)

- Token diambil dari cookie `auth_token` akun X di browser. **Token diketik saat diminta, jangan ditulis di notebook** karena dilarang ikut dikumpulkan.
- Bahasa: Inggris (`lang:en`), karena sentimen memakai VADER.

In [ ]:
QUERY = '("GTA 6" OR "GTA VI") lang:en since:2026-08-28 until:2026-10-08'
LIMIT = 400                       # ambil lebih dari 150 karena sebagian terbuang saat cleaning
RAW_FILE = "gta6_raw.csv"
print(QUERY)

("GTA 6" OR "GTA VI") lang:en since:2026-08-28 until:2026-10-08


In [ ]:
!npx -y playwright install-deps chromium > /dev/null 2>&1
!ldconfig -p | grep -q libatk-1.0 && echo "Library Chromium siap" || echo "Masih gagal"

Library Chromium siap


In [ ]:
token = getpass("Masukkan auth_token X: ").strip()
waktu_crawling = datetime.now(timezone.utc).astimezone()

# Output tweet-harvest ditampilkan baris per baris; token selalu disamarkan.
proc = subprocess.Popen(
    ["npx", "-y", "tweet-harvest@2.7.1",
     "-o", RAW_FILE, "-s", QUERY, "--tab", "LATEST", "-l", str(LIMIT), "--token", token],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line.rstrip().replace(token, "***TOKEN***"))
proc.wait()
del token

hasil = os.path.join("tweets-data", RAW_FILE)
if proc.returncode != 0 or not os.path.exists(hasil):
    raise RuntimeError("tweet-harvest gagal atau tidak menghasilkan file. Baca pesan di atas untuk penyebabnya.")
print("Waktu pengambilan data:", waktu_crawling.strftime("%d %B %Y %H:%M %Z"))

Masukkan auth_token X: ··········
npm warn deprecated rimraf@3.0.2: Rimraf versions prior to v4 are no longer supported
npm warn deprecated inflight@1.0.6: This module is not supported, and leaks memory. Do not use it. Check out lru-cache if you want a good and tested way to coalesce async requests by a key value, which is much more comprehensive and powerful.
npm warn deprecated @playwright/test@1.41.1: Please update to the latest version of Playwright to test up-to-date browsers.
npm warn deprecated glob@7.2.3: Old versions of glob are not supported, and contain widely publicized security vulnerabilities, which have been fixed in the current version. Please update. Support for old versions may be purchased (at exorbitant rates) by contacting i@izs.me
Tweet Harvest [v2.7.1]

Research by Helmi Satria
Use it for Educational Purposes only!

This script uses Chromium Browser to crawl data from Twitter with your Twitter auth token.
Please enter your Twitter auth token when prompted.

Note:

tweet-harvest menyimpan hasilnya di folder `tweets-data/`. Pindahkan salinannya ke `data/`.

In [ ]:
raw = pd.read_csv(os.path.join("tweets-data", RAW_FILE))
raw.to_csv(os.path.join(DATA_DIR, RAW_FILE), index=False)
print(raw.shape)
raw.head()

(400, 15)


,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
0,2093085709322338491,2026-10-09T04:37:28.000Z,2,🚨 ALL THE CONFIRMED DETAILS SINCE THE EXTENDED...,2108416531990290736,https://pbs.twimg.com/media/HUKZqRRW8AASaue.jpg,fieldreportgta,en,NaN,0,0,0,https://x.com/fieldreportgta/status/2108416531...,2045701372633333760,fieldreportgta
1,2108293509312635118,2026-10-09T04:37:00.000Z,0,@PapaNeedPizza @ToonStarterz Gta 6 devs wanted...,2108416415179178300,NaN,PapaNeedPizza,en,NaN,0,1,0,https://x.com/XrayPC/status/2108416415179178300,855500959,XrayPC
2,2108293509312635118,2026-10-09T04:35:59.000Z,0,@santiagoz1rk @Leashit_666777 @ToonStarterz YE...,2108416160001831190,NaN,santiagoz1rk,en,NaN,0,0,0,https://x.com/XrayPC/status/2108416160001831190,855500959,XrayPC
3,2108416089826935266,2026-10-09T04:35:42.000Z,10,Cardi B drops 'DON'T CHART' for the 'GTA VI' a...,2108416089826935266,https://pbs.twimg.com/amplify_video_thumb/2108...,NaN,en,NaN,0,0,3,https://x.com/latenightxoshow/status/210841608...,1975416413071241216,latenightxoshow
4,2108416030162784635,2026-10-09T04:35:28.000Z,0,"Weekly Recap, Oct 5–8: the GTA VI streaming re...",2108416030162784635,NaN,NaN,en,NaN,0,0,0,https://x.com/Kingfish737/status/2108416030162...,1684654476563513345,Kingfish737


## 2. Pembersihan Data

Catat angka yang dicetak di tiap langkah untuk laporan.

In [ ]:
df = raw.rename(columns={"full_text": "text", "favorite_count": "like_count"})
# Hanya kolom yang diperlukan. Username/URL tidak disimpan (hindari identitas akun).
keep = [c for c in ["id_str", "text", "created_at", "like_count", "lang"] if c in df.columns]
df = df[keep].copy()
log = {"awal": len(df)}

# a. Teks kosong / hilang
df["text"] = df["text"].astype("string").str.strip()
df = df[df["text"].notna() & (df["text"] != "")]
log["setelah_hapus_teks_kosong"] = len(df)

# b. Duplikat (ID sama atau teks sama, termasuk retweet "RT @...")
if "id_str" in df.columns:
    df = df.drop_duplicates(subset="id_str")
df["_dedup"] = df["text"].str.replace(r"^RT @\w+:\s*", "", regex=True).str.lower().str.strip()
df = df.drop_duplicates(subset="_dedup").drop(columns="_dedup")
log["setelah_hapus_duplikat"] = len(df)

# c. Kesesuaian topik: harus menyebut GTA 6 / GTA VI
topik = re.compile(r"gta\s*(?:6|vi)\b|#gta6|#gtavi", re.IGNORECASE)
df = df[df["text"].str.contains(topik)]
log["setelah_cek_topik"] = len(df)

# d. Likes harus angka valid (>= 0). Yang tidak valid jadi NaN, TIDAK diganti 0.
df["like_count"] = pd.to_numeric(df["like_count"], errors="coerce")
df.loc[df["like_count"] < 0, "like_count"] = pd.NA
log["likes_tidak_valid"] = int(df["like_count"].isna().sum())

# e. Teks bersih di kolom terpisah (teks asli tetap di kolom text)
df["clean_text"] = df["text"].apply(an.clean_text)

log

NameError: name 'raw' is not defined

In [ ]:
print("Contoh hasil pembersihan:")
df[["text", "clean_text"]].sample(5, random_state=1)

Contoh hasil pembersihan:


,text,clean_text
196,GTA 6 or me? be honest,gta or me be honest
398,@chaosprimeZ Well… I don’t think they are gonn...,well i dont think they are gonna go against it...
126,Microsoft has launched a Rockstar Games Spotli...,microsoft has launched a rockstar games spotli...
216,@PunksWorldd yeah highkey im kinda just done w...,yeah highkey im kinda just done with gta after...
337,if lana was confirmed as a radio host for gta ...,if lana was confirmed as a radio host for gta ...


**Periksa manual** beberapa postingan acak: kalau masih ada yang tidak relevan (misal spam/iklan akun), hapus dan catat jumlahnya.

In [ ]:
df[["text", "created_at", "like_count", "clean_text"]].to_csv(
    os.path.join(DATA_DIR, "gta6_clean.csv"), index=False
)
print("Tersimpan:", len(df), "postingan →", os.path.join(DATA_DIR, "gta6_clean.csv"))
assert len(df) >= 150, "Data kurang dari 150 postingan unik, ambil data lagi."


Tersimpan: 383 postingan → data/gta6_clean.csv


## Kerja Terpisah: Unggah Dataset

**Anya dan Yohanes:** setelah bagian 0 dan sel import, lewati bagian 1–2 dan jalankan sel ini. Unggah `gta6_clean.csv` dari Ilham.
Ilham tidak perlu menjalankan sel ini (file sudah ada, sel otomatis dilewati).

In [ ]:
path_clean = os.path.join(DATA_DIR, "gta6_clean.csv")
if os.path.exists(path_clean):
    print("gta6_clean.csv sudah ada, lanjut.")
else:
    print("Pilih file gta6_clean.csv dari Ilham:")
    up = files.upload()
    for name, content in up.items():
        with open(path_clean, "wb") as f:
            f.write(content)
clean = pd.read_csv(path_clean)
print(len(clean), "postingan dimuat")

Pilih file gta6_clean.csv dari Ilham:


Saving gta6_clean.csv to gta6_clean.csv
383 postingan dimuat


## 3. Pemeriksaan Manual Sentimen

**Langkah A:** ambil 20 postingan acak. File yang dibuat **hanya berisi teks**, supaya kelompok melabeli tanpa melihat prediksi sistem.

Sel di bawah mengunduh `sampel_label_manual.csv`. Buka di Excel/Sheets, isi kolom `label_manual` dengan **Positif / Negatif / Netral**, simpan sebagai CSV dengan nama yang sama.

In [ ]:
sampel = clean.sample(20, random_state=an.RANDOM_STATE)[["text"]].copy()
sampel["label_manual"] = ""
path_sampel = os.path.join(DATA_DIR, "sampel_label_manual.csv")
if os.path.exists(path_sampel):
    print("File sampel sudah ada, tidak ditimpa supaya label manual tidak hilang.")
else:
    sampel.to_csv(path_sampel, index=False)
    print("Silakan isi label manual di", path_sampel)
files.download(path_sampel)

Silakan isi label manual di data/sampel_label_manual.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Unggah kembali** file `sampel_label_manual.csv` yang sudah diisi label manual.

In [ ]:
up = files.upload()
for name, content in up.items():
    with open(path_sampel, "wb") as f:
        f.write(content)
print("Tersimpan ke", path_sampel)

Saving sampel_label_manual.csv to sampel_label_manual (1).csv
Tersimpan ke data/sampel_label_manual.csv


**Langkah B:** jalankan setelah label manual selesai diisi.

In [ ]:
sampel = pd.read_csv(path_sampel)
assert sampel["label_manual"].notna().all(), "Masih ada label_manual yang kosong."

sampel["prediksi_sistem"] = sampel["text"].apply(lambda t: an.vader_label(t)[1])
sampel.to_csv(os.path.join(DATA_DIR, "evaluasi_sentimen.csv"), index=False)

ev = an.evaluate_manual_labels(sampel)
print(f"Sesuai: {ev['n_match']} dari {ev['n']} ({ev['pct_match']:.1f}%)")
ev["mismatches"][["text", "label_manual", "prediksi_sistem"]]

Sesuai: 11 dari 20 (55.0%)


,text,label_manual,prediksi_sistem
2,@Voodoo_Hendrix @Welsh_Turkey to me GTA Online...,Netral,Positif
3,"GTA 6 this gta 6 that, I just want a loyal gir...",Netral,Positif
4,Putting Lana Del Rey and her husbands GTA 6 ra...,Netral,Positif
5,We might get GTA VI on Pc on the first week of...,Netral,Positif
6,GTA 6 will reportedly ask for your Penis size....,Negatif,Netral
8,I’m live! Streaming GTA Vice City (PS2) before...,Netral,Positif
12,@VideoGameHstry Like other game companies they...,Negatif,Positif
14,Cardi B’s new single “Don’t Chart” for GTA 6 i...,Netral,Negatif
15,I wish there was an 80s station in GTA VI so t...,Netral,Positif


Bahas 3 contoh ketidaksesuaian di laporan (misal: sarkasme, slang, konteks gim yang tidak dipahami VADER).

## 4. Cek Cepat Clustering & Regresi

**Yohanes:** setelah bagian 0, sel import, dan sel unggah dataset, langsung ke sini.

Gunakan hasil ini untuk menentukan nama tema cluster, lalu isi `CLUSTER_NAMES` di `app.py`.

In [ ]:
clean = pd.read_csv(os.path.join(DATA_DIR, "gta6_clean.csv"))
prep = an.add_sentiment(an.prepare_dataframe(clean))
_, summary, excl = an.run_clustering(prep)
print("Dikeluarkan dari clustering:", excl)
for s in summary:
    print(f"\n=== Cluster {s['cluster']} ({s['jumlah']} postingan) ===")
    print("Kata dominan:", ", ".join(s["kata_dominan"]))
    for ex in s["contoh"]:
        print("  -", ex[:200])

Dikeluarkan dari clustering: 0

=== Cluster 0 (315 postingan) ===
Kata dominan: like, just, game, play, wait
  - @xThe_Real_CJx The GTA VI of bluey comics
  - GTA 6 Countdown https://t.co/fWvPfO15fR
  - i’d rather have ps2 gta 3 then gta 6 😬

=== Cluster 1 (47 postingan) ===
Kata dominan: cardi, chart, album, dont, song
  - Cardi B’s full song “Don’t Chart” from the GTA 6 Album Track 19. 🔥🔥🔥 https://t.co/4fqtFTCJ8G
  - Cardi B’s full song “Don’t Chart” from the GTA 6 Album Track 19. https://t.co/YsjMC0eHu0
  - — Cardi B’s new single “Don’t Chart” from the GTA 6 album is OUT NOW! 🚨 https://t.co/an1Q6kUdn5

=== Cluster 2 (21 postingan) ===
Kata dominan: got, post, fucking, look, btw
  - @selfbegotten We got AIDS nazi furries before GTA 6
  - @BlkBearReloaded We got DLSS on PS5 before GTA 6
  - We got naruto in hytale before GTA 6


In [ ]:
reg = an.run_regression(prep)
print(f"Dikeluarkan (likes tidak valid): {reg['excluded']}")
print(f"Data latih/uji: {reg['n_train']}/{reg['n_test']}")
print(f"MAE: {reg['mae']:.2f}   R²: {reg['r2']:.3f}")
print("Intercept:", reg["model"].intercept_, " Koefisien [kata, hashtag]:", reg["model"].coef_)

Dikeluarkan (likes tidak valid): 0
Data latih/uji: 306/77
MAE: 516.60   R²: -0.009
Intercept: 137.0362287254941  Koefisien [kata, hashtag]: [ -2.00620022 -23.0652653 ]


## 5. Unduh Hasil

Simpan file-file ini ke folder `data/` proyek, di samping `app.py`.

In [ ]:
for f in ["gta6_raw.csv", "gta6_clean.csv", "evaluasi_sentimen.csv"]:
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        files.download(path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>